# Fluorescence Experiments Overview
``find_fluorescence_timelapse.py`` is a script designed to analyze fluorescence timelapse datasets using OpenAI's language models. The script processes datasets, extracts relevant information from log files, and classifies the data based on predefined criteria. 

This notebook is to analyse the diversity of datasets in the experiment IDs found.

In [1]:
import sys
sys.path.insert(0, "/home/ianyang/alibylite/src") # this script should be run in the alibylite environment, so we can import from src
import pandas as pd
import matplotlib.pyplot as plt
import time
from pathlib import Path
import openai
from dotenv import load_dotenv
from omero.gateway import BlitzGateway
# Load .env from the script's directory
load_dotenv(".env")

RESULTS_CSV_PATH = "results.csv"
# Load the results CSV
results_df = pd.read_csv(RESULTS_CSV_PATH)

2026-06-28 09:30:31,470 DEBUG [              omero.util.TempFileManager] (MainThread) Added file /home/ianyang/omero/tmp/.lock_testqr5bjeoz.tmp
2026-06-28 09:30:31,471 DEBUG [              omero.util.TempFileManager] (MainThread) Chose global tmpdir: /home/ianyang/omero/tmp
2026-06-28 09:30:31,471 DEBUG [              omero.util.TempFileManager] (MainThread) Using temp dir: /home/ianyang/omero/tmp/omero_ianyang/3898366


In [5]:
display(results_df)

,dataset_id,dataset_name,has_log,condition,strain_id,tf_identity,timepoints,classification,is_tf_localisation,tf_localisation_reason,reason,channels,raw_llm_response
0,2493,Ramp_0d5to0pGlc_5h_Msn2Dot6Mig1_EarlyStart_00,True,0.5% glucose to 0% glucose,1352;762;742,Msn2;Dot6;Mig1,192.0,YES,YES,"Known TF(s) detected: Msn2, Dot6, Mig1",Fluorescence channel(s) found in parsed acquis...,GFP_Z;GFP;cy5;mCherry_Z;mCherry,FLUORESCENCE: YES\nREASON: Fluorescence channe...
1,683,Aya_00,True,Aim: Strain: Comments: \r\nMicroscope se...,Comments:,UNKNOWN,180.0,YES,NO,The log shows only Brightfield and GFP with a ...,Fluorescence channel(s) found in parsed acquis...,GFP,FLUORESCENCE: YES\nREASON: Fluorescence channe...
2,2717,10xgraticule_00,True,Aim: Image of graticule slide for calibration ...,Comments: Each small division is 10um,UNKNOWN,180.0,YES,NO,This is a graticule slide calibration image in...,Fluorescence channel(s) found in parsed acquis...,GFP,FLUORESCENCE: YES\nREASON: Fluorescence channe...
3,560,pypipeline_unit_test_reconstituted_00,True,Aim: Get a successfully finished experiment wi...,365;group 1: pos001;group 2: pos002,UNKNOWN,2.0,YES,NO,The log only mentions a generic GFP channel an...,Fluorescence channel(s) found in parsed acquis...,GFP,FLUORESCENCE: YES\nREASON: Fluorescence channe...
4,561,BABY_Raff_00,True,Aim: Measure growth rate in raffinose using BA...,Comments:;group 1: YST_605;group 2: YST_625,UNKNOWN,180.0,YES,NO,This appears to be a growth-rate experiment in...,Fluorescence channel(s) found in parsed acquis...,GFP;mCherry,FLUORESCENCE: YES\nREASON: Fluorescence channe...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1820,4426,WT_Swain_in_pyruvate_3_00,True,TMRM(50nM) staining in glucose for WT swain.,NaN,UNKNOWN,1.0,NO,NO,This experiment images TMRM staining and brigh...,The only acquired channels listed are Brightfi...,UNKNOWN,FLUORESCENCE: NO\nREASON: The only acquired ch...
1821,4427,WT_Swain_in_PYRU_UN_00,True,TMRM(50nM) staining in glucose for WT swain.,NaN,UNKNOWN,1.0,NO,NO,This dataset images TMRM staining and brightfi...,The acquisition settings include a TMRM channe...,TMRM,FLUORESCENCE: YES\nREASON: The acquisition set...
1822,4428,ramp_2to0pGlc_nofluor_dIra12_pyr_pregrowth_00,True,2% glucose to 0% after 8 hours,1352;1704;1705,UNKNOWN,250.0,YES,NO,This is a growth-rate-only experiment with bri...,Fluorescence channel(s) found in parsed acquis...,cy5,FLUORESCENCE: YES\nREASON: Fluorescence channe...
1823,4451,ramp_2to0pGlc_nofluor_dIra12_pyr_pregrowth_00,True,Aim: Repeat Julien's ramp measurement for grow...,1352;1704;1705,UNKNOWN,217.0,YES,NO,This is a growth-rate-only experiment with no ...,Fluorescence channel(s) found in parsed acquis...,cy5,FLUORESCENCE: YES\nREASON: Fluorescence channe...


In [ ]:
# extract tf_localisation related experiments
results_df[
    (results_df["is_tf_localisation"] == "YES")
    & (results_df["tf_identity"].notna())
    & (results_df["tf_identity"].str.strip() != "")
]

,dataset_id,dataset_name,has_log,condition,strain_id,tf_identity,timepoints,classification,is_tf_localisation,tf_localisation_reason,reason,channels,raw_llm_response
0,2493,Ramp_0d5to0pGlc_5h_Msn2Dot6Mig1_EarlyStart_00,True,0.5% glucose to 0% glucose,1352;762;742,Msn2;Dot6;Mig1,192.0,YES,YES,"Known TF(s) detected: Msn2, Dot6, Mig1",Fluorescence channel(s) found in parsed acquis...,GFP_Z;GFP;cy5;mCherry_Z;mCherry,FLUORESCENCE: YES\nREASON: Fluorescence channe...
13,469,morph_5cham_hog1_lte1_htb2_vph1_bud3_01,True,Aim: Acquire images for morphology segmentatio...,Hog1-GFP;group 1: pos001/pos002/pos003/pos004;...,Hog1,240.0,YES,YES,Known TF(s) detected: Hog1,Fluorescence channel(s) found in parsed acquis...,GFP,FLUORESCENCE: YES\nREASON: Fluorescence channe...
62,697,SID_test_00,True,2% glucose to 0% glucose,NaN,Msn2,18.0,YES,YES,Known TF(s) detected: Msn2,Fluorescence channel(s) found in parsed acquis...,GFP;mCherry;cy5,FLUORESCENCE: YES\nREASON: Fluorescence channe...
66,615,SID_test_01,True,2% glucose to 0% glucose,NaN,Msn2,18.0,YES,YES,Known TF(s) detected: Msn2,Fluorescence channel(s) found in parsed acquis...,GFP,FLUORESCENCE: YES\nREASON: Fluorescence channe...
94,664,albicans_YPD_SCswitch_00,True,YPD to SC,crz1-GFP;group 1: pos001;group 2: pos002;group...,Crz1;Maf1;Msn2;Mig1,240.0,YES,YES,"Known TF(s) detected: Crz1, Maf1, Msn2, Mig1",Fluorescence channel(s) found in parsed acquis...,GFP,FLUORESCENCE: YES\nREASON: Fluorescence channe...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1786,4110,TFScreen_plate1_2_lowN2_00,True,high nitrogen media to low nitrogen mediaSorbi...,NaN,UNKNOWN,24.0,YES,YES,This is a transcription factor screening exper...,Fluorescence channel(s) found in parsed acquis...,GFP_Z;GFP;cy5,FLUORESCENCE: YES\nREASON: Fluorescence channe...
1787,4151,Switchearly_2to0pGlc_Msn2_dIra12_00,True,2% glucose to 0% glucose,1352;1704;1705,Msn2,192.0,YES,YES,Known TF(s) detected: Msn2,Fluorescence channel(s) found in parsed acquis...,GFP;cy5;mCherry_Z;mCherry,FLUORESCENCE: YES\nREASON: Fluorescence channe...
1790,4203,Aggregates_recovery_00,True,SC + 2% glucose to water to SC + 2% glucose,NaN,UNKNOWN,180.0,YES,YES,The experiment tracks the transcription factor...,Fluorescence channel(s) found in parsed acquis...,GFP_Z;GFP;cy5,FLUORESCENCE: YES\nREASON: Fluorescence channe...
1791,4204,TFScreen_plate1_A_B_lowN2_00,False,NaN,NaN,UNKNOWN,NaN,NO,YES,This appears to be a transcription-factor scre...,No log or acquisition metadata was attached.,UNKNOWN,FLUORESCENCE: NO\nREASON: No log or acquisitio...
